In [1]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
df = pd.DataFrame({
    'Name':       ['Asha', 'Ravi', 'Meena', 'Karan', 'Priya', 'Arjun', 'Neha', 'Dev'],
    'Department': ['HR', 'Tech', 'Sales', 'Tech', 'HR', 'Finance', 'Sales', 'Tech'],
    'Education':  ['High School', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'High School', 'PhD'],
    'City':       ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Chennai', 'Delhi', 'Bangalore', 'Mumbai'],
    'Salary':     [45000, 85000, 72000, 95000, 48000, 78000, 52000, 90000],
    'Promoted':   [0, 1, 1, 1, 0, 1, 0, 1]
})

In [3]:
print("Original DataFrame:\n", df)
print(f'\ndf type: {df.dtypes}')
print('\nUnique Values: ')
for col in ['Department', 'Education', 'City']:
  print(f'{col}: {df[col].unique()}')

Original DataFrame:
     Name Department    Education       City  Salary  Promoted
0   Asha         HR  High School     Mumbai   45000         0
1   Ravi       Tech     Bachelor      Delhi   85000         1
2  Meena      Sales       Master  Bangalore   72000         1
3  Karan       Tech          PhD     Mumbai   95000         1
4  Priya         HR     Bachelor    Chennai   48000         0
5  Arjun    Finance       Master      Delhi   78000         1
6   Neha      Sales  High School  Bangalore   52000         0
7    Dev       Tech          PhD     Mumbai   90000         1

df type: Name          object
Department    object
Education     object
City          object
Salary         int64
Promoted       int64
dtype: object

Unique Values: 
Department: ['HR' 'Tech' 'Sales' 'Finance']
Education: ['High School' 'Bachelor' 'Master' 'PhD']
City: ['Mumbai' 'Delhi' 'Bangalore' 'Chennai']


# Label Encoding


*   Label Encoding: assigns integer to each category
*   Use ONLY for target variable or tree-based models
*   NEVER for linear models implies false ordinal relationship





In [4]:
#Label Encoding
df_label = df.copy()
le = LabelEncoder()

In [5]:
# Encode Department
df_label['Department_encoded'] = le.fit_transform(df_label['Department'])
print("Label Encoding — Department:")
print(df_label[['Department', 'Department_encoded']])

# Show mapping
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\nMapping:", mapping)

# Encode multiple columns
df_label2 = df.copy()
cols_to_encode = ['Department', 'City']
encoders = {}

for col in cols_to_encode:
    enc = LabelEncoder()
    df_label2[f'{col}_encoded'] = enc.fit_transform(df_label2[col])
    encoders[col] = enc
    print(f"\n{col} mapping: {dict(zip(enc.classes_, enc.transform(enc.classes_)))}")

print("\nAfter Label Encoding:\n",
      df_label2[['Department', 'Department_encoded', 'City', 'City_encoded']])

# Problem with label encoding on nominal data
print("""
WHY Label Encoding is wrong for nominal categories:
  HR=0, Finance=1, Sales=2, Tech=3
  Model incorrectly learns: Tech > Sales > Finance > HR
  This ranking has no real meaning — use One-Hot instead.
""")

Label Encoding — Department:
  Department  Department_encoded
0         HR                   1
1       Tech                   3
2      Sales                   2
3       Tech                   3
4         HR                   1
5    Finance                   0
6      Sales                   2
7       Tech                   3

Mapping: {'Finance': np.int64(0), 'HR': np.int64(1), 'Sales': np.int64(2), 'Tech': np.int64(3)}

Department mapping: {'Finance': np.int64(0), 'HR': np.int64(1), 'Sales': np.int64(2), 'Tech': np.int64(3)}

City mapping: {'Bangalore': np.int64(0), 'Chennai': np.int64(1), 'Delhi': np.int64(2), 'Mumbai': np.int64(3)}

After Label Encoding:
   Department  Department_encoded       City  City_encoded
0         HR                   1     Mumbai             3
1       Tech                   3      Delhi             2
2      Sales                   2  Bangalore             0
3       Tech                   3     Mumbai             3
4         HR                   1    Chennai 

# One Hot Encoding
* One-Hot Encoding: creates binary column per category
* Best for nominal categories with no order
* Avoid for high-cardinality columns (100+ unique values)

In [6]:
df_ohe = df.copy()

# pandas get_dummies — simplest approach
dummies = pd.get_dummies(df_ohe['Department'], prefix='Dept')
print("One-Hot Encoded Department:\n", dummies)

# Attach to DataFrame
df_ohe = pd.concat([df_ohe, dummies], axis=1)
df_ohe.drop('Department', axis=1, inplace=True)
print("\nDataFrame after OHE on Department:\n", df_ohe.head())

# sklearn OneHotEncoder — preferred in ML pipelines
df_ohe2 = df.copy()
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids dummy variable trap
encoded = ohe.fit_transform(df_ohe2[['City']])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['City']))

print("\nsklearn OHE on City (drop first):\n", encoded_df)
print("\nCategories:", ohe.categories_)

# Dummy variable trap explanation
print("""
drop='first' avoids multicollinearity:
  If City has 4 values → 3 columns are enough
  The 4th is implied when all others are 0
  Keeping all 4 causes dummy variable trap in linear models
""")

# OHE on multiple columns at once
df_multi = df[['Department', 'City']].copy()
ohe_multi = OneHotEncoder(sparse_output=False, drop='first')
encoded_multi = ohe_multi.fit_transform(df_multi)
cols = ohe_multi.get_feature_names_out(['Department', 'City'])
df_encoded_multi = pd.DataFrame(encoded_multi, columns=cols)
print("\nOHE on Department + City:\n", df_encoded_multi)

One-Hot Encoded Department:
    Dept_Finance  Dept_HR  Dept_Sales  Dept_Tech
0         False     True       False      False
1         False    False       False       True
2         False    False        True      False
3         False    False       False       True
4         False     True       False      False
5          True    False       False      False
6         False    False        True      False
7         False    False       False       True

DataFrame after OHE on Department:
     Name    Education       City  Salary  Promoted  Dept_Finance  Dept_HR  \
0   Asha  High School     Mumbai   45000         0         False     True   
1   Ravi     Bachelor      Delhi   85000         1         False    False   
2  Meena       Master  Bangalore   72000         1         False    False   
3  Karan          PhD     Mumbai   95000         1         False    False   
4  Priya     Bachelor    Chennai   48000         0         False     True   

   Dept_Sales  Dept_Tech  
0       Fals

# Ordinal Encoding
* Ordinal Encoding: assigns integers preserving meaningful order
* Use when categories have a clear ranking

In [7]:
df_ord = df.copy()

# Define the order explicitly
education_order = [['High School', 'Bachelor', 'Master', 'PhD']]

oe = OrdinalEncoder(categories=education_order)
df_ord['Education_encoded'] = oe.fit_transform(df_ord[['Education']])

print("Ordinal Encoding — Education:")
print(df_ord[['Education', 'Education_encoded']])

# Show mapping
for i, cat in enumerate(education_order[0]):
    print(f"  {cat} → {i}")

# Compare: wrong vs right
print("\nWRONG — Label Encoding on Education (random order):")
le = LabelEncoder()
df_ord['Education_label'] = le.fit_transform(df_ord['Education'])
wrong_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(" ", wrong_mapping)

print("\nRIGHT — Ordinal Encoding (preserves order):")
right_mapping = {cat: i for i, cat in enumerate(education_order[0])}
print(" ", right_mapping)

# Multi-column ordinal encoding
satisfaction_order = [['Low', 'Medium', 'High', 'Very High']]
size_order         = [['Small', 'Medium', 'Large', 'XL']]

df_multi_ord = pd.DataFrame({
    'Satisfaction': ['High', 'Low', 'Very High', 'Medium', 'High'],
    'Size':         ['Small', 'XL', 'Large', 'Medium', 'Small']
})

oe_multi = OrdinalEncoder(categories=satisfaction_order + size_order)
df_multi_ord[['Satisfaction_enc', 'Size_enc']] = oe_multi.fit_transform(
    df_multi_ord[['Satisfaction', 'Size']]
)
print("\nMulti-column Ordinal Encoding:\n", df_multi_ord)

Ordinal Encoding — Education:
     Education  Education_encoded
0  High School                0.0
1     Bachelor                1.0
2       Master                2.0
3          PhD                3.0
4     Bachelor                1.0
5       Master                2.0
6  High School                0.0
7          PhD                3.0
  High School → 0
  Bachelor → 1
  Master → 2
  PhD → 3

WRONG — Label Encoding on Education (random order):
  {'Bachelor': np.int64(0), 'High School': np.int64(1), 'Master': np.int64(2), 'PhD': np.int64(3)}

RIGHT — Ordinal Encoding (preserves order):
  {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}

Multi-column Ordinal Encoding:
   Satisfaction    Size  Satisfaction_enc  Size_enc
0         High   Small               2.0       0.0
1          Low      XL               0.0       3.0
2    Very High   Large               3.0       2.0
3       Medium  Medium               1.0       1.0
4         High   Small               2.0       0.0


# Comparison, When to use which encoding

In [8]:
from sklearn.tree import DecisionTreeClassifier

df_model = df.copy()

# Version 1: Label Encoded (wrong for linear, ok for trees)
df_v1 = df_model.copy()
for col in ['Department', 'City', 'Education']:
    df_v1[col] = LabelEncoder().fit_transform(df_v1[col])

# Version 2: One-Hot Encoded
df_v2 = pd.get_dummies(df_model.drop('Name', axis=1),
                        columns=['Department', 'City', 'Education'])

# Version 3: Ordinal for Education, OHE for rest
df_v3 = df_model.copy()
oe = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master', 'PhD']])
df_v3['Education'] = oe.fit_transform(df_v3[['Education']])
df_v3 = pd.get_dummies(df_v3, columns=['Department', 'City'])
df_v3.drop('Name', axis=1, inplace=True)

target = 'Promoted'
for name, data in [('Label Encoded', df_v1), ('One-Hot Encoded', df_v2), ('Mixed Encoding', df_v3)]:
    features = [c for c in data.columns if c not in ['Name', 'Promoted']]
    X = data[features]
    y = data[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: Accuracy = {acc:.2f}")

# Summary table
print("""
╔══════════════════╦══════════════════════════╦════════════════════════════╗
║ Encoding         ║ Use When                 ║ Avoid When                 ║
╠══════════════════╬══════════════════════════╬════════════════════════════╣
║ Label Encoding   ║ Target variable,         ║ Nominal features in        ║
║                  ║ tree-based models        ║ linear/logistic regression ║
╠══════════════════╬══════════════════════════╬════════════════════════════╣
║ One-Hot Encoding ║ Nominal categories,      ║ High cardinality columns   ║
║                  ║ linear models            ║ (100+ unique values)       ║
╠══════════════════╬══════════════════════════╬════════════════════════════╣
║ Ordinal Encoding ║ Categories with a        ║ Nominal categories with    ║
║                  ║ clear meaningful order   ║ no natural ranking         ║
╚══════════════════╩══════════════════════════╩════════════════════════════╝
""")

Label Encoded: Accuracy = 1.00
One-Hot Encoded: Accuracy = 1.00
Mixed Encoding: Accuracy = 1.00

╔══════════════════╦══════════════════════════╦════════════════════════════╗
║ Encoding         ║ Use When                 ║ Avoid When                 ║
╠══════════════════╬══════════════════════════╬════════════════════════════╣
║ Label Encoding   ║ Target variable,         ║ Nominal features in        ║
║                  ║ tree-based models        ║ linear/logistic regression ║
╠══════════════════╬══════════════════════════╬════════════════════════════╣
║ One-Hot Encoding ║ Nominal categories,      ║ High cardinality columns   ║
║                  ║ linear models            ║ (100+ unique values)       ║
╠══════════════════╬══════════════════════════╬════════════════════════════╣
║ Ordinal Encoding ║ Categories with a        ║ Nominal categories with    ║
║                  ║ clear meaningful order   ║ no natural ranking         ║
╚══════════════════╩══════════════════════════╩═════════